# Lab 1: Building a Linear Data Pipeline with LangGraph

### Overview
In this lab, we transition from simple scripts to **orchestrated workflows**. We use **LangGraph** to manage the flow of data and **Pandas** to perform the heavy lifting.

### Architecture
Our pipeline follows the classic **ETL** pattern:
1.  **Extract**: Fetching raw data.
2.  **Transform**: Cleaning and shaping data.
3.  **Load**: Saving the finalized data for use.

### Working Principle: The Water Filtration Analogy
Imagine a city's water system:
*   **Extract**: We pump raw water from a lake (Raw CSV).
*   **Transform**: The water passes through filters to remove sand, adds minerals, and checks for safety (Data Cleaning).
*   **Load**: The clean water is sent to your tap (Final CSV/DataFrame).

In [1]:
!pip install langgraph pandas

# Lab 1: Building a Linear Data Pipeline with LangGraph

### Overview
In this lab, we transition from simple scripts to **orchestrated workflows**. We use **LangGraph** to manage the flow of data and **Pandas** to perform the heavy lifting.

### Architecture
Our pipeline follows the classic **ETL** pattern:
1.  **Extract**: Fetching raw data.
2.  **Transform**: Cleaning and shaping data.
3.  **Load**: Saving the finalized data for use.

### Working Principle: The Water Filtration Analogy
Imagine a city's water system:
*   **Extract**: We pump raw water from a lake (Raw CSV).
*   **Transform**: The water passes through filters to remove sand, adds minerals, and checks for safety (Data Cleaning).
*   **Load**: The clean water is sent to your tap (Final CSV/DataFrame).

In [2]:
import pandas as pd
import io
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

# 1. Define the 'State' (The briefcase passed between workers)
class PipelineState(TypedDict):
    raw_data: Optional[str]
    dataframe: Optional[pd.DataFrame]
    status: str
    logs: list[str]

# --- NODES (The Workers) ---

def extract_node(state: PipelineState) -> PipelineState:
    """Simulates extracting data from a source."""
    csv_content = """id,name,age,email
1,Alice,30,alice@example.com
2,Bob,,bob@example.com
3,Charlie,25,charlie@example.com
3,Charlie,25,charlie@example.com
4,David,forty,david@example.com
"""
    print("[Agent: Extract] -> Fetching raw CSV data...")
    return {
        **state,
        "raw_data": csv_content,
        "logs": state["logs"] + ["Data extracted from source."]
    }

def transform_node(state: PipelineState) -> PipelineState:
    """Cleans the data using 3 operations."""
    print("[Agent: Transform] -> Starting cleaning operations...")
    df = pd.read_csv(io.StringIO(state["raw_data"]))

    # Operation 1: Remove Duplicates
    df = df.drop_duplicates()

    # Operation 2: Handle Missing Values (Fill missing age with mean or 0)
    # First, force numeric to handle 'forty' error
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['age'] = df['age'].fillna(df['age'].mean())

    # Operation 3: Fix Data Types
    df['id'] = df['id'].astype(int)

    return {
        **state,
        "dataframe": df,
        "logs": state["logs"] + ["Data cleaned: duplicates removed, types fixed, nulls filled."]
    }

def load_node(state: PipelineState) -> PipelineState:
    """Saves/Displays the final result."""
    print("[Agent: Load] -> Finalizing data for production...")
    return {
        **state,
        "status": "Completed",
        "logs": state["logs"] + ["Data loaded into final destination."]
    }

# --- ORCHESTRATION (The Manager) ---

workflow = StateGraph(PipelineState)

workflow.add_node("extract", extract_node)
workflow.add_node("transform", transform_node)
workflow.add_node("load", load_node)

workflow.set_entry_point("extract")
workflow.add_edge("extract", "transform")
workflow.add_edge("transform", "load")
workflow.add_edge("load", END)

app = workflow.compile()

# Execute
initial_state = {"raw_data": None, "dataframe": None, "status": "Started", "logs": []}
final_output = app.invoke(initial_state)

[Agent: Extract] -> Fetching raw CSV data...
[Agent: Transform] -> Starting cleaning operations...
[Agent: Load] -> Finalizing data for production...


### Agent Reasoning: Behind the Scenes

| Step | What the Agent Did | Why it did it | Logic/Decision |
| :--- | :--- | :--- | :--- |
| **Extract** | Read a string-based CSV. | We cannot process data that isn't in memory. | Decision: Grab the rawest form to ensure no data is lost before cleaning. |
| **Transform** | Dropped row 4 (duplicate), converted 'forty' to NaN, then filled it. | Duplicates skew metrics; non-numeric ages break calculations. | Decision: Use `errors='coerce'` to safely handle 'bad' strings without crashing the pipeline. |
| **Load** | Validated the state and marked 'Completed'. | To signal to the next system that the data is ready for the UI or Database. | Decision: Finalize only after confirming the DataFrame is no longer null. |

In [3]:
print("--- Final Cleaned Dataset ---")
display(final_output['dataframe'])

print("\n--- Execution Logs ---")
for log in final_output['logs']:
    print(f"- {log}")

--- Final Cleaned Dataset ---


,id,name,age,email
0,1,Alice,30.0,alice@example.com
1,2,Bob,27.5,bob@example.com
2,3,Charlie,25.0,charlie@example.com
4,4,David,27.5,david@example.com



--- Execution Logs ---
- Data extracted from source.
- Data cleaned: duplicates removed, types fixed, nulls filled.
- Data loaded into final destination.


### Key Learnings
1.  **State Management**: By passing a `state` object, every node knows exactly what the previous node finished.
2.  **Immutability**: We don't just change variables; we return a new state, making the pipeline easier to debug.
3.  **Resilience**: Using `pd.to_numeric(errors='coerce')` allows our agent to 'decide' to turn garbage data into a format (NaN) that we can programmatically fix later.